<a href="https://colab.research.google.com/github/amankiitg/LLM_Prod/blob/main/Reditt_Text_Clustering_and_Topic_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Text Clustering and Topic Modeling</h1>
<i>Clustering documents using a wide variety of language models.</i>



### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [ ]:
!pip install asyncpraw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.4/196.4 kB 3.9 MB/s eta 0:00:00


In [ ]:
import asyncio
import asyncpraw
import nest_asyncio
import json
import numpy as np
import re
import string

# --------------------
# Text cleaning helper
# --------------------
def clean_text(text):
    """Cleans text by removing URLs, punctuation, and lowercasing."""
    text = re.sub(r'http\S+|https\S+', '', text)   # Remove URLs
    text = text.replace('\n', ' ')                 # Remove newlines
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    return text.lower().strip()

def extract_comment_bodies(comments):
    """Recursively extracts bodies and scores for thresholding."""
    bodies, scores = [], []
    for comment in comments:
        bodies.append(comment['body'])
        scores.append(comment['score'])
        if comment.get('replies'):
            child_bodies, child_scores = extract_comment_bodies(comment['replies'])
            bodies.extend(child_bodies)
            scores.extend(child_scores)
    return bodies, scores

# --------------------
# Main Scraper
# --------------------
async def scrape():
    async with asyncpraw.Reddit(
        client_id="glnTvdUs1KFycKy5vEsOig",
        client_secret="ET1IgiyCztIxhXSwnFIYbTs3eq2nRQ",
        user_agent="Topic Modeler"
    ) as reddit:

        url = "https://www.reddit.com/r/ChatGPT/comments/1mkae1l/gpt5_ama_with_openais_sam_altman_and_some_of_the/"
        submission = await reddit.submission(url=url)

        async def extract_comment(comment):
            return {
                "author": str(comment.author),
                "body": comment.body,
                "score": comment.score,
                "replies": [await extract_comment(reply) async for reply in comment.replies]
            }

        # Build post object
        post_data = {
            "title": submission.title,
            "author": str(submission.author),
            "score": submission.score,
            "url": submission.url,
            "selftext": submission.selftext,
            "comments": []
        }

        # Load comments
        await submission.comments.replace_more(limit=None)
        for top_comment in submission.comments:
            post_data["comments"].append(await extract_comment(top_comment))

        # Flatten scores for median threshold
        _, all_scores = extract_comment_bodies(post_data["comments"])
        median_score = np.median(all_scores) if all_scores else 0

        # --------------------
        # Filter + transform comments
        # --------------------
        def filter_and_transform(comments):
            """Filter by median score and create title/abstract."""
            result = []
            for c in comments:
                if c['score'] >= median_score:
                    result.append({
                        "title": clean_text(c['body'])[:60],   # First 60 chars as title
                        "abstract": clean_text(c['body']),     # Full cleaned body
                        "score": c['score'],
                        "author": c['author']
                    })
                if c.get('replies'):
                    result.extend(filter_and_transform(c['replies']))
            return result

        post_data["comments"] = filter_and_transform(post_data["comments"])
        post_data["median_score_threshold"] = median_score

        # Dump JSON
        with open("reddit_post_filtered.json", "w", encoding="utf-8") as f:
            json.dump(post_data, f, indent=2, ensure_ascii=False)

        print("✅ Filtered post data saved to reddit_post_filtered.json")

# --------------------
# Run safely with nest_asyncio
# --------------------
nest_asyncio.apply()
asyncio.run(scrape())


✅ Filtered post data saved to reddit_post_filtered.json


/tmp/ipython-input-3660819704.py:65: DeprecationWarning: Using CommentForest as an asynchronous iterator has been deprecated and will be removed in a future version.
  post_data["comments"].append(await extract_comment(top_comment))
/tmp/ipython-input-3660819704.py:49: DeprecationWarning: Using CommentForest as an asynchronous iterator has been deprecated and will be removed in a future version.
  "replies": [await extract_comment(reply) async for reply in comment.replies]


# **Load Reditt Data**

In [14]:
import json

with open("/content/reddit_post_filtered.json", "r", encoding="utf-8") as f:
    reddit_data = json.load(f)

# Recursive extractor for title and abstract
def extract_title_abstract(comments):
    result = []
    for c in comments:
        result.append({
            "title": c["title"],
            "abstract": c["abstract"]
        })
        if c.get("replies"):
            result.extend(extract_title_abstract(c["replies"]))
    return result

# Extract only title and abstract
titles_and_abstracts = extract_title_abstract(reddit_data["comments"])

# Display first 10
for item in titles_and_abstracts[:10]:
    print(item)

# # Extract metadata and convert to standard list
abstracts = [t['abstract'] for t in titles_and_abstracts]
titles = [t['title'] for t in titles_and_abstracts]

{'title': 'i see that em dash', 'abstract': 'i see that em dash'}
{'title': 'its not an em dash — its a pause that demands attention', 'abstract': 'its not an em dash — its a pause that demands attention'}
{'title': 'can you mark the other cohosts their answers are buried and ', 'abstract': 'can you mark the other cohosts their answers are buried and unhighlighted and the only way you can find them at present is by checking each profiles comment history manually   currently the filter for answered misses every other cohost reply other than sam altman making the ama much harder to navigate  or provide some other way to see them thanks'}
{'title': 'only uopenai can i think theres a limit to the number of par', 'abstract': 'only uopenai can i think theres a limit to the number of participants'}
{'title': 'ah thats a bother perhaps sticky links to them in a comment ', 'abstract': 'ah thats a bother perhaps sticky links to them in a comment after the ama ends its kind of silly that theres b

## **BERTopic: A Modular Topic Modeling Framework**

In [15]:
!pip install -q bertopic openai datasets datamapplot

In [16]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration, OpenAI
from transformers import pipeline
from wordcloud import WordCloud
import openai
from copy import deepcopy
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from google.colab import userdata
import datamapplot
import re


def find_best_k(embeddings, k_min=2, k_max=15, random_state=42):
    """Return best k based on silhouette score"""
    best_score = -1
    best_k = k_min
    for k in range(k_min, k_max + 1):
        kmeans = KMeans(n_clusters=k, random_state=random_state).fit(embeddings)
        score = silhouette_score(embeddings, kmeans.labels_)
        if score > best_score:
            best_score = score
            best_k = k
    print(f"✅ Best k for KMeans: {best_k} with silhouette {best_score:.3f}")
    return best_k

# ==================== Helper Functions ====================
def topic_differences(model, original_topics, nr_topics=5):
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]
    return df

def create_wordcloud(model, topic, save_path=None):
    """Generate a wordcloud for a given topic if it exists."""
    topic_words = model.get_topic(topic)
    if not topic_words:  # Returns False if topic does not exist
        print(f"⚠️ Topic {topic} does not exist. Skipping wordcloud.")
        return

    plt.figure(figsize=(10,5))
    text = {word: value for word, value in topic_words}
    wc = WordCloud(background_color="white", max_words=1000, width=1600, height=800)
    wc.generate_from_frequencies(text)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.close()


def save_document_visualization(model, documents, reduced_embeddings, save_path=None):
    """
    Visualize BERTopic documents and save as HTML.

    Args:
        model: BERTopic model
        documents: list of documents/abstracts
        reduced_embeddings: reduced embeddings used for visualization
        save_path: path to save HTML file
    """
    fig = model.visualize_documents(
        documents,
        reduced_embeddings=reduced_embeddings,
        width=1200,
        hide_annotations=True
    )
    fig.update_layout(font=dict(size=8))

    if save_path:
        fig.write_html(save_path)
        print(f"✅ Document visualization saved as HTML: {save_path}")

    plt.close()



def get_cluster_info(model, documents):
    """Return a dict with each cluster's top sentences and topic labels"""
    clusters = {}
    topics = model.get_document_info(documents)
    for topic_id in topics['Topic'].unique():
        if topic_id == -1:
            continue
        cluster_docs = topics[topics.Topic==topic_id]['Document'].tolist()
        label = model.get_topic(topic_id)
        clusters[topic_id] = {
            "top_sentences": cluster_docs[:5],  # top 5 representative sentences
            "topic_label": model.get_topic_info().loc[model.get_topic_info().Topic==topic_id, "Name"].values[0]
        }
    return clusters



# ==================== Config ====================
embedding_models = {
    "gte-small": "thenlper/gte-small",
    "multilingual-e5-large": "intfloat/multilingual-e5-large-instruct"
}

dim_reduction_models = {
    # "umap": UMAP(n_components=5, min_dist=0.0, metric="cosine", random_state=42),
    "pca": PCA(n_components=5, random_state=42)
}

clustering_models = {
    "hdbscan": HDBSCAN(min_cluster_size=10, metric="euclidean", cluster_selection_method="eom"),
    "dbscan": DBSCAN(eps=0.01, min_samples=5, metric="euclidean"),
    "kmeans": KMeans(n_clusters=10, random_state=42)
}

In [17]:
def save_document_visualization_png(model, documents, reduced_embeddings, save_path=None):
    """
    Visualize BERTopic documents and save as PNG with arrow annotations.

    Args:
        model: BERTopic model
        documents: list of documents/abstracts
        reduced_embeddings: reduced embeddings used for visualization
        save_path: path to save PNG file
    """
    import numpy as np
    import matplotlib.pyplot as plt

    # Save as PNG using matplotlib
    if save_path:
        # Ensure the save path has .png extension
        if not save_path.lower().endswith('.png'):
            save_path = save_path.rsplit('.', 1)[0] + '.png'

        # Create matplotlib version
        plt.figure(figsize=(12, 8))

        # Get topic assignments and create color map
        topic_assignments = np.array(model.topics_)
        unique_topics = np.unique(topic_assignments)
        colors = plt.cm.Set3(np.linspace(0, 1, len(unique_topics)))
        color_map = {topic: colors[i] for i, topic in enumerate(unique_topics)}

        # Plot points
        for topic in unique_topics:
            mask = topic_assignments == topic
            if topic == -1:  # Outliers
                plt.scatter(reduced_embeddings[mask, 0], reduced_embeddings[mask, 1],
                          c='gray', alpha=0.6, s=20, label='Outliers')
            else:
                plt.scatter(reduced_embeddings[mask, 0], reduced_embeddings[mask, 1],
                          c=[color_map[topic]], alpha=0.7, s=30, label=f'Topic {topic}')

        # Randomly select 20% of topics for labeling
        import random
        non_outlier_topics = [t for t in unique_topics if t != -1]
        num_topics_to_label = max(1, int(len(non_outlier_topics) * 0.2))
        topics_to_label = random.sample(non_outlier_topics, num_topics_to_label)

        # Add arrows pointing to cluster centers (only for selected topics)
        for topic in unique_topics:
            if topic != -1 and topic in topics_to_label:  # Only label selected topics
                mask = topic_assignments == topic
                if np.any(mask):
                    center_x = np.mean(reduced_embeddings[mask, 0])
                    center_y = np.mean(reduced_embeddings[mask, 1])

                    # Get topic words and wrap text
                    topic_words = model.get_topic(topic)
                    topic_label = ", ".join([word for word, _ in topic_words[:3]])

                    # Wrap text to limit width (approximately 15 characters per line)
                    import textwrap
                    wrapped_label = "\n".join(textwrap.wrap(topic_label, width=15))
                    final_label = f"Topic {topic}\n{wrapped_label}"

                    # Calculate direction from overall center to cluster center
                    overall_center_x = np.mean(reduced_embeddings[:, 0])
                    overall_center_y = np.mean(reduced_embeddings[:, 1])

                    direction_x = center_x - overall_center_x
                    direction_y = center_y - overall_center_y

                    # Normalize and extend (back to original distance)
                    length = np.sqrt(direction_x**2 + direction_y**2)
                    if length > 0:
                        direction_x /= length
                        direction_y /= length

                    # Place text at original distance (2.5)
                    text_distance = 2.5
                    text_x = center_x + direction_x * text_distance
                    text_y = center_y + direction_y * text_distance

                    # Add arrow and text
                    plt.annotate(final_label,
                               xy=(center_x, center_y),
                               xytext=(text_x, text_y),
                               arrowprops=dict(arrowstyle='->', color='black', lw=2),
                               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
                               fontsize=8, ha='center', va='center')

        plt.title('Document Visualization with Topic Clusters', fontsize=14)
        plt.xlabel('Dimension 1', fontsize=12)
        plt.ylabel('Dimension 2', fontsize=12)
        plt.grid(True, alpha=0.3)

        # Save matplotlib version
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        # plt.close()
        print(f"Document visualization saved as PNG: {save_path}")

    return None

In [18]:
import matplotlib.pyplot as plt

def save_topic_differences_png(topic_model, original_topics, label, model_dir):
    """Save topic differences as PNG instead of just displaying."""
    diff_df = topic_differences(topic_model, original_topics)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    ax.table(
        cellText=diff_df.values,
        colLabels=diff_df.columns,
        cellLoc='center',
        loc='center'
    )
    plt.tight_layout()
    save_path = os.path.join(model_dir, f"{label}_topic_differences.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"📊 Saved {label} topic differences → {save_path}")


In [ ]:
results = {}

output_dir = "bertopic_outputs"
os.makedirs(output_dir, exist_ok=True)

# ==================== Main Loop ====================
for emb_name, emb_model_name in embedding_models.items():
    print(f"\n==== Embedding Model: {emb_name} ====")

    # Load embedding model
    embedding_model = SentenceTransformer(emb_model_name)
    embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

    for dim_name, dim_model in dim_reduction_models.items():
        reduced_embeddings = dim_model.fit_transform(embeddings)

        for clust_name, clust_model in clustering_models.items():

            # If KMeans, find optimal k
            if clust_name == "kmeans":
                best_k = find_best_k(reduced_embeddings, k_min=2, k_max=15)
                clust_model = KMeans(n_clusters=best_k, random_state=42)

            model_name = f"{emb_name}_{dim_name}_{clust_name}"
            print(f"\n--- {model_name} ---")

            # Create output folder
            model_dir = os.path.join(output_dir, model_name)
            os.makedirs(model_dir, exist_ok=True)

            # Build BERTopic model
            topic_model = BERTopic(
                embedding_model=embedding_model,
                umap_model=dim_model if dim_name=="umap" else None,
                hdbscan_model=clust_model if clust_name=="hdbscan" else None,
                verbose=False
            )

            # Fit model
            topics, probs = topic_model.fit_transform(abstracts, embeddings)

            # Get topic info
            info = topic_model.get_topic_info()

            print('\nBERT Topics')
            display(info)

            # Exclude outlier topic (-1)
            num_clusters = len(info[info.Topic != -1])
            print(f"Number of clusters (excluding outliers): {num_clusters}")

            # Save original topic representations
            original_topics = deepcopy(topic_model.topic_representations_)

            # ===== Update representations =====
            topic_model.update_topics(abstracts, representation_model=KeyBERTInspired())

            # Show topic differences
            print('\nKeyBERTInspired')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "KeyBERTInspired", model_dir)


            topic_model.update_topics(abstracts, representation_model=MaximalMarginalRelevance(diversity=0.5))

            print('\nMMR')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "MMR", model_dir)


            max_len = 400
            truncated_docs = [doc[:max_len] for doc in abstracts]

            generator = pipeline("text2text-generation", model="google/flan-t5-small")
            rep_t5 = TextGeneration(generator, prompt="Topic: [KEYWORDS]\nDocs: [DOCUMENTS]")
            topic_model.update_topics(truncated_docs, representation_model=rep_t5)

            print('\nT5')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "T5", model_dir)



            prompt = """
            I have a topic that contains the following comments from reditt thread:
            [COMMENTS]

            The topic is described by the following keywords: [KEYWORDS]

            Based on the information above, extract a short topic label in the following format:
            topic: <short topic label>
            """

            # Update our topic representations using GPT-3.5
            client = openai.OpenAI(api_key=userdata.get('openaikey'))
            representation_model = OpenAI(
                client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt
            )
            topic_model.update_topics(abstracts, representation_model=representation_model)

            print('\nOpen AI')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "Open AI", model_dir)


            # ===== Save visualizations =====
            save_document_visualization_png(topic_model, abstracts, reduced_embeddings, save_path=os.path.join(model_dir, "document_vis.png"))

            save_document_visualization(topic_model, abstracts, reduced_embeddings, save_path=os.path.join(model_dir, "document_vis.html"))

            topic_model.update_topics(abstracts, top_n_words=500)

            # Wordclouds for first 5 topics
            for topic_id in range(5):
                create_wordcloud(topic_model, topic_id, save_path=os.path.join(model_dir, f"wordcloud_topic{topic_id}.png"))

            # ===== Cluster info and topic labels =====
            clusters_info = get_cluster_info(topic_model, abstracts)
            # Save cluster info as JSON
            # import json
            # with open(os.path.join(model_dir, "clusters_info.json"), "w", encoding="utf-8") as f:
            #     json.dump(clusters_info, f, indent=2, ensure_ascii=False)

            # Print top topic labels
            # for cid, cinfo in clusters_info.items():
            #     print(f"Cluster {cid}: Label = {cinfo['topic_label']}")
            #     for sent in cinfo['top_sentences']:
            #         print(f"  - {sent}")

            # Path to save the cluster info
            txt_file_path = os.path.join(model_dir, "clusters_info.txt")

            with open(txt_file_path, "w", encoding="utf-8") as f:
                for cid, cinfo in clusters_info.items():
                    f.write(f"Cluster {cid}: Label = {cinfo['topic_label']}\n")
                    for sent in cinfo['top_sentences']:
                        f.write(f"  - {sent}\n")
                    f.write("\n")  # Add a newline between clusters

            print(f"✅ Cluster info saved to {txt_file_path}")


            results[model_name] = topic_model



==== Embedding Model: gte-small ====


Batches:   0%|          | 0/87 [00:00<?, ?it/s]


--- gte-small_pca_hdbscan ---

BERT Topics


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1021,-1_to_and_it_the,"[to, and, it, the, of, for, is, that, 4o, in]",[i believe it is important to have both 4o and...
1,0,163,0_context_window_32k_128k,"[context, window, 32k, 128k, for, plus, tokens...",[will you ever give plus users a decent contex...
2,1,120,1_models_legacy_the_model,"[models, legacy, the, model, users, to, you, h...",[since you already have them available for pro...
3,2,89,2_ding_spaghetti_word_smith,"[ding, spaghetti, word, smith, eating, blueber...",[yes it sees everything and i mean everything ...
4,3,86,3_safety_the_censorship_for,"[safety, the, censorship, for, you, that, be, ...",[can you do something about the filter surely ...
5,4,84,4_back_bring_4o_please,"[back, bring, 4o, please, he, friend, my, best...","[please bring back 4o, please bring 4o back, p..."
6,5,82,5_gpt5_gpt_it_to,"[gpt5, gpt, it, to, the, out, gpt4, and, did, he]",[gpt5 has the same problem as every previous m...
7,6,81,6_plus_limits_limit_unlimited,"[plus, limits, limit, unlimited, to, the, for,...",[as a plus subscriber i paid to work freely — ...
8,7,77,7_voice_standard_mode_advanced,"[voice, standard, mode, advanced, cove, the, k...",[i appreciate the innovation truly but the dec...
9,8,70,8_4o_it_and_to,"[4o, it, and, to, like, was, personality, the,...",[i really hope they go back on this decision i...


Number of clusters (excluding outliers): 45

KeyBERTInspired


...

📊 Saved T5 topic differences → bertopic_outputs/multilingual-e5-large_pca_kmeans/T5_topic_differences.png

Open AI


,Topic,Original,Updated
0,0,gpt5 | gpt | to | it | the,Conversations with GPT-5 and its capabilities
1,1,and | the | to | of | it,AI language models and emotional response tren...
2,2,context | window | 32k | 128k | for,Natural Language Processing and Chat Length li...
3,3,4o | to | it | and | the,Creative Writing Process
4,4,4o | back | bring | please | release,Request for Release of Friend


📊 Saved Open AI topic differences → bertopic_outputs/multilingual-e5-large_pca_kmeans/Open AI_topic_differences.png


2025-08-21 20:09:01,939 - BERTopic - WARNING: Note that extracting more than 100 words from a sparse can slow down computation quite a bit.


Document visualization saved as PNG: bertopic_outputs/multilingual-e5-large_pca_kmeans/document_vis.png
✅ Document visualization saved as HTML: bertopic_outputs/multilingual-e5-large_pca_kmeans/document_vis.html
Cluster 18: Label = 18_em_dashes_dash_emdashes
  - i see that em dash
  - its not an em dash — its a pause that demands attention
  - its sloppier also are you going to address the emdashes
  - ive been using emdashes for 20 years alt0151 they arent the problem the fact that people dont normally use them and that theyve become a signal for aigenerated content is the problem tell the gpt not to use them
  - just put it in your custom instructions if you dont want them used   i love using emdashes myself and have been using them for some years now ever since i switched to a mac since you can just press optionshifthyphen to get it but now if im in a scenario when i dont want folks thinking ai did my writing i just make a point to replace all the emdashes with hyphens
Cluster 5: La

# Only one config

In [19]:
# ==================== Config ====================
embedding_models = {
    # "gte-small": "thenlper/gte-small",
    "multilingual-e5-large": "intfloat/multilingual-e5-large-instruct"
}

dim_reduction_models = {
    "umap": UMAP(n_components=5, min_dist=0.0, metric="cosine", random_state=42),
    # "pca": PCA(n_components=5, random_state=42)
}

clustering_models = {
    "hdbscan": HDBSCAN(min_cluster_size=10, metric="euclidean", cluster_selection_method="eom"),
    # "dbscan": DBSCAN(eps=0.01, min_samples=5, metric="euclidean"),
    # "kmeans": KMeans(n_clusters=10, random_state=42)
}

output_dir = "bertopic_outputs"
os.makedirs(output_dir, exist_ok=True)

# ==================== Main Loop ====================
for emb_name, emb_model_name in embedding_models.items():
    print(f"\n==== Embedding Model: {emb_name} ====")

    # Load embedding model
    embedding_model = SentenceTransformer(emb_model_name)
    embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

    for dim_name, dim_model in dim_reduction_models.items():
        reduced_embeddings = dim_model.fit_transform(embeddings)

        for clust_name, clust_model in clustering_models.items():

            # If KMeans, find optimal k
            if clust_name == "kmeans":
                best_k = find_best_k(reduced_embeddings, k_min=2, k_max=15)
                clust_model = KMeans(n_clusters=best_k, random_state=42)

            model_name = f"{emb_name}_{dim_name}_{clust_name}"
            print(f"\n--- {model_name} ---")

            # Create output folder
            model_dir = os.path.join(output_dir, model_name)
            os.makedirs(model_dir, exist_ok=True)

            # Build BERTopic model
            topic_model = BERTopic(
                embedding_model=embedding_model,
                umap_model=dim_model if dim_name=="umap" else None,
                hdbscan_model=clust_model if clust_name=="hdbscan" else None,
                verbose=False
            )

            # Fit model
            topics, probs = topic_model.fit_transform(abstracts, embeddings)

            # Get topic info
            info = topic_model.get_topic_info()

            print('\nBERT Topics')
            display(info)

            # Exclude outlier topic (-1)
            num_clusters = len(info[info.Topic != -1])
            print(f"Number of clusters (excluding outliers): {num_clusters}")

            # Save original topic representations
            original_topics = deepcopy(topic_model.topic_representations_)

            # ===== Update representations =====
            topic_model.update_topics(abstracts, representation_model=KeyBERTInspired())

            # Show topic differences
            print('\nKeyBERTInspired')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "KeyBERTInspired", model_dir)


            topic_model.update_topics(abstracts, representation_model=MaximalMarginalRelevance(diversity=0.5))

            print('\nMMR')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "MMR", model_dir)


            max_len = 400
            truncated_docs = [doc[:max_len] for doc in abstracts]

            generator = pipeline("text2text-generation", model="google/flan-t5-small")
            rep_t5 = TextGeneration(generator, prompt="Topic: [KEYWORDS]\nDocs: [DOCUMENTS]")
            topic_model.update_topics(truncated_docs, representation_model=rep_t5)

            print('\nT5')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "T5", model_dir)



            prompt = """
            I have a topic that contains the following comments from reditt thread:
            [COMMENTS]

            The topic is described by the following keywords: [KEYWORDS]

            Based on the information above, extract a short topic label in the following format:
            topic: <short topic label>
            """

            # Update our topic representations using GPT-3.5
            client = openai.OpenAI(api_key=userdata.get('openaikey'))
            representation_model = OpenAI(
                client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt
            )
            topic_model.update_topics(abstracts, representation_model=representation_model)

            print('\nOpen AI')
            display(topic_differences(topic_model, original_topics))
            save_topic_differences_png(topic_model, original_topics, "Open AI", model_dir)


            # ===== Save visualizations =====
            save_document_visualization_png(topic_model, abstracts, reduced_embeddings, save_path=os.path.join(model_dir, "document_vis.png"))

            save_document_visualization(topic_model, abstracts, reduced_embeddings, save_path=os.path.join(model_dir, "document_vis.html"))

            topic_model.update_topics(abstracts, top_n_words=500)

            # Wordclouds for first 5 topics
            for topic_id in range(5):
                create_wordcloud(topic_model, topic_id, save_path=os.path.join(model_dir, f"wordcloud_topic{topic_id}.png"))

            # ===== Cluster info and topic labels =====
            clusters_info = get_cluster_info(topic_model, abstracts)
            # Save cluster info as JSON
            # import json
            # with open(os.path.join(model_dir, "clusters_info.json"), "w", encoding="utf-8") as f:
            #     json.dump(clusters_info, f, indent=2, ensure_ascii=False)

            # Print top topic labels
            # for cid, cinfo in clusters_info.items():
            #     print(f"Cluster {cid}: Label = {cinfo['topic_label']}")
            #     for sent in cinfo['top_sentences']:
            #         print(f"  - {sent}")

            # Path to save the cluster info
            txt_file_path = os.path.join(model_dir, "clusters_info.txt")

            with open(txt_file_path, "w", encoding="utf-8") as f:
                for cid, cinfo in clusters_info.items():
                    f.write(f"Cluster {cid}: Label = {cinfo['topic_label']}\n")
                    for sent in cinfo['top_sentences']:
                        f.write(f"  - {sent}\n")
                    f.write("\n")  # Add a newline between clusters

            print(f"✅ Cluster info saved to {txt_file_path}")


            results[model_name] = topic_model



==== Embedding Model: multilingual-e5-large ====


Batches:   0%|          | 0/87 [00:00<?, ?it/s]


--- multilingual-e5-large_umap_hdbscan ---

BERT Topics


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1109,-1_the_to_it_and,"[the, to, it, and, is, you, for, of, that, 4o]",[sam and the rest at openai while this questi...
1,0,233,0_and_the_to_it,"[and, the, to, it, of, that, for, in, gpt4o, w...",[dear openai team i’m writing as a longtime a...
2,1,141,1_context_window_32k_128k,"[context, window, 32k, 128k, for, is, the, plu...",[hello thanks for answering this one id like...
3,2,88,2_release_4o_back_bring,"[release, 4o, back, bring, please, you, we, pl...",[release 4o release 4o release 4o release 4o r...
4,3,77,3_models_legacy_users_plus,"[models, legacy, users, plus, model, have, pro...",[is it possible to have the legacy models also...
5,4,75,4_4o_plus_back_users,"[4o, plus, back, users, for, to, pay, month, p...",[we are looking into letting plus users to con...
6,5,74,5_you_ama_they_questions,"[you, ama, they, questions, answer, about, thi...",[did you see the negative feedback that people...
7,6,68,6_the_for_censorship_filter,"[the, for, censorship, filter, its, and, flagg...",[agreed that sounds frustrating you should be ...
8,7,63,7_voice_standard_mode_advanced,"[voice, standard, mode, advanced, keep, the, c...",[i appreciate the innovation truly but the dec...
9,8,59,8_agreed_yes_yessss_agreeing,"[agreed, yes, yessss, agreeing, this, exactly,...","[agreed, agreed, yes this]"


Number of clusters (excluding outliers): 41

KeyBERTInspired


...

,Topic,Original,Updated
0,0,and | the | to | it | of,Natural Language Processing advancements and e...
1,1,context | window | 32k | 128k | for,Context window size in Gemini Pro model
2,2,release | 4o | back | bring | please,Request for Release and Support in Bringing Ba...
3,3,models | legacy | users | plus | model,Legacy models and user loyalty
4,4,4o | plus | back | users | for,Subscription Service Optimization


📊 Saved Open AI topic differences → bertopic_outputs/multilingual-e5-large_umap_hdbscan/Open AI_topic_differences.png
Document visualization saved as PNG: bertopic_outputs/multilingual-e5-large_umap_hdbscan/document_vis.png


2025-08-21 21:03:53,693 - BERTopic - WARNING: Note that extracting more than 100 words from a sparse can slow down computation quite a bit.


✅ Document visualization saved as HTML: bertopic_outputs/multilingual-e5-large_umap_hdbscan/document_vis.html
✅ Cluster info saved to bertopic_outputs/multilingual-e5-large_umap_hdbscan/clusters_info.txt


NameError: name 'results' is not defined

In [21]:
prompt = """
I have a topic that contains the following comments from reditt thread:
[COMMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <short topic label>
"""

# Update our topic representations using GPT-3.5
client = openai.OpenAI(api_key=userdata.get('openaikey'))
representation_model = OpenAI(
    client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt
)
topic_model.update_topics(abstracts, representation_model=representation_model)

print('\nOpen AI')
display(topic_differences(topic_model, original_topics))


Open AI


,Topic,Original,Updated
0,0,and | the | to | it | of,Natural Language Processing AI Models
1,1,context | window | 32k | 128k | for,Context window for Gemini Pro tokens
2,2,release | 4o | back | bring | please,Request for bringing back a feature or service
3,3,models | legacy | users | plus | model,legacy models and user support in new updates
4,4,4o | plus | back | users | for,Discussion on introducing a paid subscription ...


In [24]:
topic_model.get_topic_info()['Representation'].values

array([list(['Discussion on GPT-5 Models']),
       list(['Natural Language Processing AI Models']),
       list(['Context window for Gemini Pro tokens']),
       list(['Request for bringing back a feature or service']),
       list(['legacy models and user support in new updates']),
       list(['Discussion on introducing a paid subscription model for users to access premium features on a platform.']),
       list(['User AMA Feedback and Responses']),
       list(['Internet Censorship and Safety Measures']),
       list(['Voice Mode Standardization']),
       list(['Agreement and Excitement on Deceptive Outcome of Reditt Thread']),
       list(["Understanding OpenAI's mission and decisions"]),
       list(['GPT-5 Model Performance and Alignment Instructions']),
       list(['Expressing gratitude and receiving support on social media']),
       list(['Improvements in GPT-5 for Creative Writing, Image, and Audio Creation']),
       list(['Importance of Teamwork and Cooperation']),
     

In [8]:
from google.colab import files
import shutil

# Folder you want to download
folder_path = "all_summaries"

# Make a zip archive
shutil.make_archive("all_summaries", 'zip', folder_path)

# Download the zip
files.download("all_summaries.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import os
import itertools
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import json

# ==================== Config ====================
embedding_models = {
    "gte-small": "thenlper/gte-small",
    "multilingual-e5-large": "intfloat/multilingual-e5-large-instruct"
}

dim_reduction_models = {
    "umap": "umap",
    "pca": "pca"
}

clustering_models = {
    "hdbscan": "hdbscan",
    "dbscan": "dbscan",
    "kmeans": "kmeans"
}

# Root folders
output_root = "bertopic_outputs"
summary_root = "all_summaries"
os.makedirs(summary_root, exist_ok=True)

def generate_summary_page(config_name):
    config_path = os.path.join(output_root, config_name)

    # Representation model images (1x4)
    rep_models = ["KeyBERTInspired", "MMR", "T5", "Open AI"]
    rep_images = [os.path.join(config_path, f"{rm}_topic_differences.png") for rm in rep_models]

    # Wordclouds (top 5 topics)
    wc_images = [os.path.join(config_path, f"wordcloud_topic{i}.png") for i in range(5)]

    # Cluster visualization (full width, double size)
    doc_vis = os.path.join(config_path, "document_vis.png")

    # Number of clusters (excluding outliers)
    cluster_file = os.path.join(config_path, "clusters_info.json")
    num_clusters = None
    if os.path.exists(cluster_file):
        with open(cluster_file, "r", encoding="utf-8") as f:
            clusters_info = json.load(f)
        num_clusters = sum(1 for cid in clusters_info if cid != "-1")

    # Create figure
    fig = plt.figure(figsize=(12, 18))  # increased height for cluster visual

    # --- Row 1: Topic representation differences (1x4) ---
    for i, img_path in enumerate(rep_images):
        if os.path.exists(img_path):
            ax = plt.subplot2grid((5, 4), (0, i), colspan=1, rowspan=1)  # 5 rows, 4 cols, first row
            ax.imshow(mpimg.imread(img_path))
            ax.set_title(rep_models[i], fontsize=10)
            ax.axis("off")

    # --- Row 2: Wordclouds (top 5 topics) ---
    for i, img_path in enumerate(wc_images):
        if os.path.exists(img_path):
            ax = plt.subplot2grid((5, 5), (1, i))  # row 2, 5 columns
            ax.imshow(mpimg.imread(img_path))
            ax.set_title(f"Topic {i}", fontsize=10)
            ax.axis("off")

    # --- Row 3: Cluster visualization (double size) ---
    ax = plt.subplot2grid((5, 1), (2, 0), rowspan=3)  # spans last 3 rows
    if os.path.exists(doc_vis):
        ax.imshow(mpimg.imread(doc_vis))
        title_text = "Cluster Visualization"
        if num_clusters is not None:
            title_text += f" | #Clusters (excluding outliers) = {num_clusters}"
        ax.set_title(title_text, fontsize=12)
    ax.axis("off")

    # Adjust layout
    plt.subplots_adjust(wspace=0, hspace=0.2, top=0.93)

    # Supertitle for configuration
    plt.suptitle(f"Configuration: {config_name}", fontsize=16, y=0.96)

    # Save figure
    save_path = os.path.join(summary_root, f"{config_name}_summary.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"Saved summary: {save_path}")

# --- Loop over all configs ---
for emb, red, clus in itertools.product(embedding_models.keys(),
                                        dim_reduction_models.keys(),
                                        clustering_models.keys()):
    config_name = f"{emb}_{red}_{clus}"
    generate_summary_page(config_name)

print(f"\nAll summary images are stored in folder: {summary_root}")


Saved summary: all_summaries/gte-small_umap_hdbscan_summary.png
Saved summary: all_summaries/gte-small_umap_dbscan_summary.png
Saved summary: all_summaries/gte-small_umap_kmeans_summary.png
Saved summary: all_summaries/gte-small_pca_hdbscan_summary.png
Saved summary: all_summaries/gte-small_pca_dbscan_summary.png
Saved summary: all_summaries/gte-small_pca_kmeans_summary.png
Saved summary: all_summaries/multilingual-e5-large_umap_hdbscan_summary.png
Saved summary: all_summaries/multilingual-e5-large_umap_dbscan_summary.png
Saved summary: all_summaries/multilingual-e5-large_umap_kmeans_summary.png
Saved summary: all_summaries/multilingual-e5-large_pca_hdbscan_summary.png
Saved summary: all_summaries/multilingual-e5-large_pca_dbscan_summary.png
Saved summary: all_summaries/multilingual-e5-large_pca_kmeans_summary.png

All summary images are stored in folder: all_summaries


In [11]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import json

# ==================== Config ====================
embedding_models = ["gte-small", "multilingual-e5-large"]
dim_reduction_models = ["umap", "pca"]
clustering_models = ["hdbscan", "dbscan", "kmeans"]

# Root folders
output_root = "bertopic_outputs"
summary_root = "all_summaries"
os.makedirs(summary_root, exist_ok=True)

def get_num_clusters(config_path):
    cluster_file = os.path.join(config_path, "clusters_info.json")
    if os.path.exists(cluster_file):
        with open(cluster_file, "r", encoding="utf-8") as f:
            clusters_info = json.load(f)
        return sum(1 for cid in clusters_info if cid != "-1")
    return None

def generate_combined_cluster_plot(embedding_name):
    fig, axes = plt.subplots(len(dim_reduction_models), len(clustering_models),
                             figsize=(18, 8))  # rows: dim reduction, cols: clustering
    for i, dim_red in enumerate(dim_reduction_models):
        for j, clus in enumerate(clustering_models):
            config_name = f"{embedding_name}_{dim_red}_{clus}"
            config_path = os.path.join(output_root, config_name)
            doc_vis_path = os.path.join(config_path, "document_vis.png")

            ax = axes[i, j]

            if os.path.exists(doc_vis_path):
                img = mpimg.imread(doc_vis_path)
                ax.imshow(img)
                num_clusters = get_num_clusters(config_path)
                title_text = f"{embedding_name.upper()} + {dim_red.upper()} + {clus.upper()}"
                if num_clusters is not None:
                    title_text += f" | #Clusters={num_clusters}"
                ax.set_title(title_text, fontsize=10)
            else:
                ax.text(0.5, 0.5, f"Missing {config_name}", ha='center', va='center')

            ax.axis("off")

    plt.tight_layout()
    save_path = os.path.join(summary_root, f"{embedding_name}_all_dimred_clusters.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"Saved combined cluster plot: {save_path}")

# --- Loop over embeddings ---
for emb in embedding_models:
    generate_combined_cluster_plot(emb)

print(f"\nAll combined cluster images are stored in folder: {summary_root}")


Saved combined cluster plot: all_summaries/gte-small_all_dimred_clusters.png
Saved combined cluster plot: all_summaries/multilingual-e5-large_all_dimred_clusters.png

All combined cluster images are stored in folder: all_summaries
